# Feature engineering

## 1. Notebook setup

### 1.1. Imports

In [ ]:
import json
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier

### 1.2. Run configuration

In [ ]:
# Run control
GENERATE_AGGREGATE_FEATURES = True

# File paths
CV_FOLD_DATASET            = '../data/tmp/03-preprocessed-cv-folds.pkl'
WINNING_HYPERPARAMETERS    = '../data/results/03-winning-hyperparameters.pkl'
AGGREGATE_FEATURES_DATASET = '../data/tmp/04-aggregate-features.pkl'

## 2. Data preparation

### 2.1. Data loading

In [ ]:
# Load preprocessed training cross-validation folds
with open(CV_FOLD_DATASET, 'rb') as f:
    cv_folds = pd.read_pickle(f)

for i, fold in enumerate(cv_folds):
    print(f'CV fold {i + 1}: {list(fold.keys())}')

### 2.2. Metadata loading

In [ ]:
# Load dataset metadata
with open("../data/schema.json", "r") as f:
    metadata = json.load(f)

# Extract feature lists
features = metadata['features']
categorical_features = metadata['categorical_features']
continuous_features = metadata['continuous_features']
high_cardinality_feature = metadata['high_cardinality_feature']
label = metadata['label']

print(f'Categorical features: {", ".join(categorical_features)}')
print(f'Continuous features: {", ".join(continuous_features)}')
print(f'High cardinality feature: {high_cardinality_feature}')
print(f'Label: {label}')

## 3. Binned features

### 3.1. Candidates

In [ ]:
cv_folds[0]['x_train'].info()

In [ ]:
if GENERATE_AGGREGATE_FEATURES:

    candidate_feature_folds = []

    for i, fold in enumerate(cv_folds):

        candidate_feature_folds.append({
            'x_train': {},
            'x_validation': {}
        })

        for feature in fold['x_train'].columns:
            if len(fold["x_train"][feature].unique()) > 4:

                print(f'\nBinning feature "{feature}" in CV fold {i + 1}...')

                feature_bins = pd.qcut(fold['x_train'][feature], q=4, labels=False, duplicates='drop')
                candidate_feature_folds[i]['x_train'][f'{feature}_bins'] = feature_bins

                feature_bins = pd.qcut(fold['x_validation'][feature], q=4, labels=False, duplicates='drop')
                candidate_feature_folds[i]['x_validation'][f'{feature}_bins'] = feature_bins

                for agg_feature in continuous_features:

                    print(f' Aggregating feature "{agg_feature}" by "{feature}_bins" in CV fold {i + 1}...')

                    if feature != agg_feature:

                        new_feature_name = f'{feature}_bins_{agg_feature}_median'
                        candidate_feature_folds[i]['x_train'][new_feature_name] = fold['x_train'].groupby(candidate_feature_folds[i]['x_train'][f'{feature}_bins'])[agg_feature].transform('median')
                        candidate_feature_folds[i]['x_validation'][new_feature_name] = fold['x_validation'].groupby(candidate_feature_folds[i]['x_validation'][f'{feature}_bins'])[agg_feature].transform('median')

            elif len(fold["x_train"][feature].unique()) <= 4:

                for agg_feature in continuous_features:

                    print(f' Aggregating feature "{agg_feature}" by "{feature}" in CV fold {i + 1}...')

                    if feature != agg_feature:

                        new_feature_name = f'{feature}_{agg_feature}_median'
                        candidate_feature_folds[i]['x_train'][new_feature_name] = fold['x_train'].groupby(fold['x_train'][feature])[agg_feature].transform('median')
                        candidate_feature_folds[i]['x_validation'][new_feature_name] = fold['x_validation'].groupby(fold['x_validation'][feature])[agg_feature].transform('median')

    with open(AGGREGATE_FEATURES_DATASET, 'wb') as f:
        pd.to_pickle(candidate_feature_folds, f)


else:
    with open(AGGREGATE_FEATURES_DATASET, 'rb') as f:
        candidate_feature_folds = pd.read_pickle(f)

### 3.2. Candidate selection

In [ ]:
# with open(WINNING_HYPERPARAMETERS, 'rb') as f:
#     winning_hyperparameters = pd.read_pickle(f)

#     print(f'Winning hyperparameters: {winning_hyperparameters}')

for i, fold in enumerate(cv_folds):

    x_train = fold['x_train']
    y_train = fold['y_train']
    x_validation = fold['x_validation']
    y_validation = fold['y_validation']

    model = HistGradientBoostingClassifier(max_iter=100, random_state=42)

    model.fit(x_train, y_train)

    validation_predictions = model.predict_proba(x_validation)

